## Imports core libraries for Spark ETL:

In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date, regexp_extract, trim, when

Creates a SparkSession named "Data Lake Builder" running locally.

In [13]:
spark = SparkSession.builder \
    .appName("Data Lake Builder") \
    .master("spark://spark-master:7077") \
    .config("spark.executor.instances", "1") \
    .config("spark.executor.cores", "2") \
    .config("spark.executor.memory", "5g") \
    .config("spark.sql.shuffle.partitions", "12") \
    .getOrCreate()

In [14]:
RAW_ZONE = "/datalake/Raw"
CLEANSED_ZONE = "/datalake/Silver"
GOLD_ZONE = "/datalake/Gold"

Loads CSV files from the RAW_ZONE into Spark DataFrames with headers and automatic schema inference for further processing.

In [15]:
accounts_raw = spark.read.csv(f"{RAW_ZONE}/accounts.csv", header=True, inferSchema=True)
details_raw = spark.read.csv(f"{RAW_ZONE}/account_details.csv", header=True, inferSchema=True)
Person_raw = spark.read.csv(f"{RAW_ZONE}/Person.csv", header=True, inferSchema=True)
profile_raw = spark.read.csv(f"{RAW_ZONE}/person_profile.csv", header=True, inferSchema=True)
iden_raw = spark.read.csv(f"{RAW_ZONE}/person_iden.csv", header=True, inferSchema=True)

### Cleansing Data

In [16]:
# Accounts
accounts_clean = (
    accounts_raw
    .toDF(*[c.strip() for c in accounts_raw.columns])
    .withColumn("Update_Status_Date", to_date(col("Date"), "dd-MMM-yy"))
    .withColumnRenamed("Acc no", "Acc_No")
    .drop("Date")
    .dropDuplicates()
)

# account_details
details_clean = (
    details_raw
    .toDF(*[c.strip() for c in details_raw.columns])
    .withColumn("Update_Type_Date", to_date(col("Date"), "dd-MMM-yy"))
    .withColumnRenamed("Acc no", "Acc_No")
    .drop("Date")
    .dropDuplicates()
)

# Person
person_clean = (
    Person_raw
    .toDF(*[c.strip() for c in Person_raw.columns])
    .withColumnRenamed("Acc no", "Acc_No")
    .dropDuplicates()
)

# person_profile
profile_clean = (
    profile_raw
    .toDF(*[c.strip() for c in profile_raw.columns])
    .withColumn("Update_Name_Date", to_date(col("Date"), "dd-MMM-yy"))
    .drop("Date")
    .dropDuplicates()
)

# person_iden
iden_clean = (
    iden_raw
    .toDF(*[c.strip() for c in iden_raw.columns])
    .withColumn("Update_Id_Date", to_date(col("Date"), "dd-MMM-yy"))
    .withColumn("id_number", trim(regexp_extract(col("Id"), r"^([A-Za-z0-9]+)", 1)))
    .withColumn("id_type", regexp_extract(col("Id"), r"\(([^)]+)\)", 1))
    .withColumn("id_type", when(col("id_type") == "", None).otherwise(col("id_type")))
    .drop("Date", "Id")
    .dropDuplicates()
)

### Write data in a silver layer 

In [17]:
accounts_clean.write.mode("overwrite").parquet(f"{CLEANSED_ZONE}/accounts/")
details_clean.write.mode("overwrite").parquet(f"{CLEANSED_ZONE}/account_details/")
person_clean.write.mode("overwrite").parquet(f"{CLEANSED_ZONE}/Person_silver/")
profile_clean.write.mode("overwrite").parquet(f"{CLEANSED_ZONE}/person_profile/")
iden_clean.write.mode("overwrite").parquet(f"{CLEANSED_ZONE}/person_iden/")

Writes the Gold dimension table to Parquet format in the GOLD_ZONE with overwrite mode and partitions the data by Matching_date for optimized querying.

In [18]:
dim_account_gold = accounts_clean.join(
    details_clean, 
    (accounts_clean.Acc_No == details_clean.Acc_No) & 
    (accounts_clean.Update_Status_Date == details_clean.Update_Type_Date), 
    "left"
).select(
    accounts_clean["Acc_No"],
    details_clean["type"].alias("Account_Type"),
    accounts_clean["Status"].alias("Account_Status"),
    accounts_clean["Update_Status_Date"].alias("Matching_date")
)

In [19]:
dim_account_gold.write\
            .mode("overwrite")\
            .partitionBy("Matching_date")\
            .parquet(f"{GOLD_ZONE}/dim_account/")


Joins person profile with person data to build a Gold level person dimension table, then saves it as partitioned Parquet files in the GOLD_ZONE by Update_Name_Date.

In [20]:
dim_person_gold = profile_clean.join(
    person_clean,
    profile_clean.Person == person_clean.Person,
    "left"
).select(
    person_clean["Acc_no"],
    profile_clean["Person"],
    profile_clean["Name"],
    profile_clean["Update_Name_Date"]
)

In [21]:
dim_person_gold.write\
            .mode("overwrite")\
            .partitionBy("Update_Name_Date")\
            .parquet(f"{GOLD_ZONE}/dim_person/")


Writes the identity cleaned dataset to the GOLD_ZONE as partitioned Parquet files using Update_Id_Date for optimized storage and querying.

In [22]:
iden_clean.write\
            .mode("overwrite")\
            .partitionBy("Update_Id_Date")\
            .parquet(f"{GOLD_ZONE}/dim_Identity/")

In [23]:
# Stops the active Spark session and releases all associated resources.

spark.stop()